# Explorer et préparer les avis sur des médicaments avec Spark

Ce TP prépare des données pour l'apprentissage automatique avec Spark : des avis de patients sur des médicaments, qui serviront à prédire si un avis est positif. Vous y explorez les données (statistiques descriptives, valeurs manquantes, invalides et aberrantes, doublons), les nettoyez, puis construisez une à une les variables qu'attend un modèle de Spark ML : variables numériques mises à l'échelle, variables catégorielles encodées, texte découpé en mots puis vectorisé (TF-IDF, Word2Vec), sélection de variables et réduction de dimension. Vous finissez par un pipeline de préparation complet, nettoyage compris, entraîné, sauvegardé, rechargé et appliqué à des avis tels que lus.

Chaque exercice s'appuie sur le précédent : faites-les dans l'ordre. Spark tourne ici en mode local, sur les cœurs de la machine de Colab. Les diapositives travaillent sur la partie test du jeu de données (53 766 avis) ; ce TP prend la partie entraînement, trois fois plus grande : vos chiffres seront donc différents, mais les anomalies sont les mêmes.

## Installation et données

Les données sont le [Drug Review Dataset](https://archive.ics.uci.edu/dataset/462/drug+review+dataset+drugs+com) (Gräßer et Kallumadi, UCI Machine Learning Repository, 2018, licence CC BY 4.0) : des avis publiés sur Drugs.com de 2008 à 2017, en anglais. Le fichier `drugsComTrain_raw.tsv` (84 Mo) en est la partie entraînement. Chaque ligne est un avis, avec un identifiant sans nom de colonne, le médicament (`drugName`), l'affection soignée (`condition`), le texte de l'avis (`review`), la note de 1 à 10 (`rating`), la date de publication écrite en anglais (`date`, par exemple « May 20, 2012 ») et le nombre de lecteurs qui ont trouvé l'avis utile (`usefulCount`).

In [ ]:
!pip install pyspark
!wget -q -nc https://github.com/shuuchuu/datasets/raw/refs/heads/main/drug-reviews/drugsComTrain_raw.tsv

In [ ]:
import html
import os
import sys

from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import (
  IDF,
  PCA,
  Binarizer,
  CountVectorizer,
  OneHotEncoder,
  QuantileDiscretizer,
  RegexTokenizer,
  SQLTransformer,
  StandardScaler,
  StopWordsRemover,
  StringIndexer,
  UnivariateFeatureSelector,
  VectorAssembler,
  Word2Vec,
)
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

La session tourne en local sur deux cœurs. Les agrégations répartissent leurs résultats sur `spark.sql.shuffle.partitions` partitions : les 200 par défaut ralentiraient une si petite table, quatre suffisent. Les processus Python que Spark lance pour exécuter du code Python (l'UDF de l'exercice 7) prennent l'interpréteur de ce notebook, `sys.executable`, qui a pandas et pyarrow : hors de Colab, Spark prendrait sinon le `python3` du `PATH`, qui peut ne pas les avoir.

In [ ]:
os.environ["PYSPARK_PYTHON"] = sys.executable
spark = (
  SparkSession.builder.appName("avis-preparation")
  .master("local[2]")
  .config("spark.sql.shuffle.partitions", "4")
  .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

## 1. Charger les avis

Lisez `drugsComTrain_raw.tsv` dans le `DataFrame` `avis` : un fichier séparé par des tabulations, avec une ligne d'en-tête, dont Spark devine les types (`inferSchema`). Un avis peut tenir sur plusieurs lignes, entre guillemets, et ses propres guillemets y sont doublés : choisissez les options de lecture en conséquence. Renommez la première colonne, `_c0`, en `id`.

Combien de lignes compte `avis` ? Et combien Spark en compterait-il sans l'option des lignes multiples ? Affichez le schéma, puis le nombre de partitions de `avis` (`avis.rdd.getNumPartitions()`) : pourquoi n'y en a-t-il qu'une ? Répartissez enfin `avis` sur quatre partitions et mettez-le en cache, pour que les exercices suivants ne relisent pas le fichier.


**Documentation utile**

- [pyspark.sql.DataFrame.repartition](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.repartition.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les options sont celles des diapositives : `sep="\t"`, `header=True`, `inferSchema=True`, `multiLine=True` et `escape='"'`. Un fichier lu avec `multiLine=True` ne se découpe pas : Spark ne sait pas où commence une ligne sans lire tout ce qui la précède. `cache` ne fait rien tout de suite : le `count` qui suit remplit le cache.

```python
avis = spark.read.csv(
  "drugsComTrain_raw.tsv",
  sep=...,
  header=True,
  inferSchema=True,
  multiLine=...,
  escape=...,
).withColumnRenamed("_c0", "id")
print(spark.read.csv("drugsComTrain_raw.tsv", sep="\t", header=True).count())
avis.printSchema()
print(avis.rdd.getNumPartitions())
avis = avis.repartition(...).cache()
avis.count()
```

### Solution

In [ ]:
avis = spark.read.csv(
  "drugsComTrain_raw.tsv",
  sep="\t",
  header=True,
  inferSchema=True,
  multiLine=True,
  escape='"',
).withColumnRenamed("_c0", "id")
print(spark.read.csv("drugsComTrain_raw.tsv", sep="\t", header=True).count())
avis.printSchema()
print(avis.rdd.getNumPartitions())
avis = avis.repartition(4).cache()
avis.count()
# 161 297 avis, contre 197 054 lignes sans multiLine : chaque saut de ligne d'un avis
# commencerait une nouvelle ligne. Le fichier est lu d'un seul bloc (une partition),
# car Spark ne peut pas couper un fichier dont une ligne peut s'étendre sur plusieurs.

## 2. Statistiques descriptives

Affichez les statistiques descriptives de `rating` et de `usefulCount` (`summary`), puis le nombre d'avis par note, dans l'ordre des notes.

Que dit la comparaison de la moyenne et de la médiane de `usefulCount` sur sa distribution ? Quelle part des avis a une note de 7 ou plus, les avis positifs que le modèle apprendra à reconnaître ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

`summary()` donne les quartiles, là où `describe()` s'arrête au minimum et au maximum. Pour la part des avis positifs, filtrez par une condition SQL (`"rating >= 7"`) et divisez par le nombre total d'avis (161 297).

```python
avis.select(..., ...).summary().show()
avis.groupBy(...).count().orderBy(...).show()
avis.filter(...).count() / avis.count()
```

### Solution

In [ ]:
avis.select("rating", "usefulCount").summary().show()
avis.groupBy("rating").count().orderBy("rating").show()
avis.filter("rating >= 7").count() / avis.count()
# usefulCount : une moyenne de 28 pour une médiane de 16, et un maximum de 1 291 : une
# distribution asymétrique, avec une longue traîne d'avis très lus. Les notes se
# concentrent aux extrémités (10 pour 50 989 avis, 1 pour 21 619) ; 66 % des avis ont
# une note de 7 ou plus, les classes ne sont donc pas équilibrées.

## 3. Valeurs manquantes

Comptez, en une seule passe sur les données, les valeurs `NULL` de chaque colonne de `avis`, puis les chaînes vides (ou faites d'espaces) des colonnes `drugName`, `condition` et `review`. Quelle colonne a des valeurs manquantes, et combien ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Une liste en compréhension construit une agrégation par colonne, toutes passées au même `select` : `F.count_if(condition)` compte les lignes où la condition est vraie. Une chaîne vide se teste par `F.trim(colonne) == ""`.

```python
avis.select([F.count_if(F.col(c).isNull()).alias(c) for c in ...]).show()
avis.select(
  [F.count_if(...).alias(c) for c in ["drugName", "condition", "review"]]
).show()
```

### Solution

In [ ]:
avis.select([F.count_if(F.col(c).isNull()).alias(c) for c in avis.columns]).show()
avis.select(
  [F.count_if(F.trim(c) == "").alias(c) for c in ["drugName", "condition", "review"]]
).show()
# Seule condition a des valeurs manquantes : 899 NULL. Aucune chaîne vide.

## 4. Valeurs invalides

Affichez les dix conditions les plus fréquentes parmi celles qui contiennent `</span>` ou qui commencent par une minuscule, avec leur nombre d'avis. Que s'est-il passé à la collecte des données ?

Comptez ensuite, en une passe, les avis concernés par chacune de ces règles : condition qui contient `</span>`, condition qui commence par une minuscule, note hors de l'intervalle de 1 à 10, date illisible au format `MMMM d, yyyy`, avis qui contient une entité HTML (`&#039;`, `&amp;`, `&quot;`…). Gardez dans la variable `invalide` la condition qui repère une condition invalide : l'exercice 7 s'en sert.

In [ ]:
# Votre code ici

### Conseils supplémentaires

`F.col("condition").contains("</span>")` teste une sous-chaîne, `rlike("^[a-z]")` une expression régulière (ici, une minuscule en tête). Deux conditions se combinent par `|` (ou), `&` (et), `~` (non). `F.try_to_date(colonne, format)` rend `NULL` pour une date qu'il ne sait pas lire ; une entité HTML se repère par l'expression `&#?\w+;`.

```python
invalide = F.col("condition").contains(...) | F.col("condition").rlike(...)
avis.filter(invalide).groupBy("condition").count().orderBy(F.desc("count")).show(
  10, truncate=False
)
avis.select(
  F.count_if(...).alias("span"),
  F.count_if(...).alias("tronquee"),
  F.count_if(~F.col("rating").between(..., ...)).alias("note_hors_1_10"),
  F.count_if(F.try_to_date(...).isNull()).alias("date_illisible"),
  F.count_if(F.col("review").rlike(...)).alias("entites_html"),
).show()
```

### Solution

In [ ]:
invalide = F.col("condition").contains("</span>") | F.col("condition").rlike("^[a-z]")
avis.filter(invalide).groupBy("condition").count().orderBy(F.desc("count")).show(
  10, truncate=False
)
avis.select(
  F.count_if(F.col("condition").contains("</span>")).alias("span"),
  F.count_if(F.col("condition").rlike("^[a-z]")).alias("tronquee"),
  F.count_if(~F.col("rating").between(1, 10)).alias("note_hors_1_10"),
  F.count_if(F.try_to_date("date", "MMMM d, yyyy").isNull()).alias("date_illisible"),
  F.count_if(F.col("review").rlike(r"&#?\w+;")).alias("entites_html"),
).show()
# La collecte a mal découpé les pages : 900 conditions sont un bout de la page
# (« 3</span> users found this comment helpful. »), 2 453 sont tronquées au début
# (« ibromyalgia » pour Fibromyalgia, 1 791 avis). Les notes et les dates sont
# toutes valides. 105 313 avis contiennent une entité HTML, à décoder avant de
# découper le texte en mots.

## 5. Valeurs aberrantes

Calculez le premier et le troisième quartile de `usefulCount` avec `approxQuantile`, à 1 % d'erreur relative, puis le seuil de la boîte à moustaches, $Q_3 + 1{,}5 \times (Q_3 - Q_1)$. Combien d'avis le dépassent ? Affichez les cinq avis les plus utiles (sans le texte) : sont-ce des erreurs à retirer, ou des valeurs extrêmes à garder ? Que remarquez-vous sur leurs médicaments et leurs dates ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

`approxQuantile` prend la colonne, la liste des probabilités (`[0.25, 0.75]`) et l'erreur relative (`0.01`), et rend une liste de valeurs à dépaqueter.

```python
q1, q3 = avis.approxQuantile(..., [...], ...)
seuil = ...
print(q1, q3, seuil, avis.filter(F.col("usefulCount") > seuil).count())
avis.orderBy(F.desc(...)).select(
  "drugName", "condition", "rating", "date", "usefulCount"
).show(5)
```

### Solution

In [ ]:
q1, q3 = avis.approxQuantile("usefulCount", [0.25, 0.75], 0.01)
seuil = q3 + 1.5 * (q3 - q1)
print(q1, q3, seuil, avis.filter(F.col("usefulCount") > seuil).count())
avis.orderBy(F.desc("usefulCount")).select(
  "drugName", "condition", "rating", "date", "usefulCount"
).show(5)
# Q1 = 6, Q3 = 35 : 12 016 avis dépassent le seuil de 78,5. Ce sont de vrais avis,
# anciens et sur des médicaments courants (un antidépresseur, un stérilet) : on les
# garde, et le logarithme de usefulCount réduira leur poids. Les avis vont par
# paires : Sertraline et Zoloft, même date, même nombre de lecteurs (exercice
# suivant).

## 6. Doublons

Un même avis apparaît sous le nom commercial du médicament et sous le nom de sa molécule (Zoloft et Sertraline). Regroupez les avis par texte, date, note et `usefulCount`, et gardez les groupes de plus d'un avis, avec la liste des noms de médicaments de chacun. Combien d'avis sont en trop ? Affichez les trois groupes aux avis les plus utiles.

Pourquoi ces doublons gêneraient-ils l'évaluation d'un modèle ?


**Documentation utile**

- [pyspark.sql.functions.collect_set](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.collect_set.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`agg` calcule plusieurs agrégations à la fois : `F.count("*")` et `F.collect_set("drugName")`, qui rend les valeurs distinctes du groupe. Un groupe de $n$ avis en a $n - 1$ en trop : sommez `F.col("n") - 1`.

```python
cles = ["review", "date", "rating", "usefulCount"]
doublons = (
  avis.groupBy(cles)
  .agg(F.count("*").alias("n"), F.collect_set(...).alias("noms"))
  .filter(...)
)
doublons.agg(F.sum(...)).show()
doublons.orderBy(F.desc("usefulCount")).select(
  "noms", "rating", "date", "usefulCount"
).show(3, truncate=False)
```

### Solution

In [ ]:
cles = ["review", "date", "rating", "usefulCount"]
doublons = (
  avis.groupBy(cles)
  .agg(F.count("*").alias("n"), F.collect_set("drugName").alias("noms"))
  .filter("n > 1")
)
doublons.agg(F.sum(F.col("n") - 1)).show()
doublons.orderBy(F.desc("usefulCount")).select(
  "noms", "rating", "date", "usefulCount"
).show(3, truncate=False)
# 48 671 avis en trop, près d'un sur trois. Laissés en place, ils mettraient un même
# avis dans l'entraînement et dans le test : le modèle serait évalué sur des avis
# qu'il a déjà vus, et son score serait trop optimiste.

## 7. Nettoyer et mesurer la fuite

Le code ci-dessous construit `propres` comme les diapositives : sans les doublons de l'exercice 6, avec les conditions invalides (la variable `invalide` de l'exercice 4) et manquantes remplacées par `"inconnue"`, le texte des avis sans ses guillemets de tête et de fin et ses entités HTML décodées (une UDF Python), une colonne `jour`, la date convertie en `date`, et l'étiquette `positif`, $1$ pour une note de 7 ou plus (`Binarizer`). Il coupe ensuite `propres` en `entrainement` (80 %) et `test` (20 %), avec la graine 42. Exécutez-le : combien d'avis reste-t-il, combien ont une condition `"inconnue"`, et combien sont positifs ?

Mesurez ensuite la fuite qu'évite le dédoublonnage. Coupez `avis`, sans le dédoublonner, de la même façon (80 %, 20 %, graine 42), et comptez les avis de sa partie test dont les clés de l'exercice 6 (`cles` : texte, date, note et `usefulCount`) se retrouvent dans sa partie entraînement, par une semi-jointure. Combien en trouvez-vous, et combien entre `test` et `entrainement` ? La réponse à la question de l'exercice 6 se vérifie-t-elle ?


**Documentation utile**

- [pyspark.sql.DataFrame.join](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html)

In [ ]:
decoder = F.udf(lambda texte: html.unescape(texte[1:-1]), "string")
propres = (
  avis.dropDuplicates(cles)
  .withColumn("condition", F.when(~invalide, F.col("condition")))
  .withColumn("review", decoder("review"))
  .withColumn("jour", F.to_date("date", "MMMM d, yyyy"))
  .fillna({"condition": "inconnue"})
)
etiquette = Binarizer(threshold=6.0, inputCol="rating", outputCol="positif")
propres = etiquette.transform(propres).cache()
print(
  propres.count(),
  propres.filter("condition = 'inconnue'").count(),
  propres.filter("positif = 1").count(),
)
entrainement, test = propres.randomSplit([0.8, 0.2], seed=42)
entrainement.cache()
print(entrainement.count(), test.count())

# Votre code ici

### Conseils supplémentaires

`randomSplit` rend une liste de `DataFrame`, à dépaqueter. `gauche.join(droite, cles, "left_semi")` garde les lignes de `gauche` qui ont une correspondance dans `droite` sur les colonnes `cles`, sans leur ajouter de colonne ni les répéter : son `count()` est le nombre d'avis du test déjà vus.

```python
avis_entrainement, avis_test = avis.randomSplit(..., seed=42)
print(avis_test.count(), avis_test.join(..., cles, ...).count())
print(test.join(..., cles, "left_semi").count())
```

### Solution

In [ ]:
avis_entrainement, avis_test = avis.randomSplit([0.8, 0.2], seed=42)
print(avis_test.count(), avis_test.join(avis_entrainement, cles, "left_semi").count())
print(test.join(entrainement, cles, "left_semi").count())
# Le code fourni garde 112 626 avis, dont 2 915 de condition inconnue et 74 754
# positifs ; 90 292 pour l'entraînement, 22 334 pour le test. Sans dédoublonnage,
# 15 382 des 32 146 avis du test (près d'un sur deux) sont aussi dans l'entraînement :
# le modèle serait évalué en partie sur des avis qu'il a appris. Après, aucun.

## 8. Créer des variables

Le `SQLTransformer` `variables` ci-dessous crée les trois variables des diapositives. Ajoutez à sa requête deux colonnes : `jour_semaine`, le jour de la semaine de `jour` (la fonction SQL `dayofweek`, 1 pour le dimanche), et `nb_mots`, le nombre de mots de l'avis, séparés par des espaces. Appliquez-le à `entrainement`, dans `avec_variables`, et affichez le minimum, la médiane et le maximum des deux nouvelles colonnes. La part d'avis positifs (la moyenne de `positif`) varie-t-elle selon le jour de la semaine ?

Découpez ensuite `usefulCount` en quatre classes, aux quartiles, avec un `QuantileDiscretizer` appris sur `entrainement`, vers `utile_quartile`. Quelles bornes a-t-il trouvées ? La part d'avis positifs varie-t-elle d'une classe à l'autre ?


**Documentation utile**

- [pyspark.sql.functions.dayofweek](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.dayofweek.html)
- [pyspark.sql.functions.split](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.split.html)
- [pyspark.sql.functions.size](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.size.html)

In [ ]:
variables = SQLTransformer(
  statement="""
    SELECT *, log1p(usefulCount) AS log_utile, length(review) AS longueur,
           year(jour) AS annee
    FROM __THIS__
  """
)

# Votre code ici

### Conseils supplémentaires

Les fonctions SQL ont les noms de celles de `F` : `dayofweek(jour)`, et `size(split(review, ' '))`, le nombre de morceaux de l'avis coupé aux espaces. La part d'avis positifs d'un groupe est la moyenne de `positif` : `agg(F.avg("positif"))`. `QuantileDiscretizer(numBuckets=4, ...)` est un estimateur : son modèle, un `Bucketizer`, donne ses bornes par `getSplits()`.

```python
variables = SQLTransformer(
  statement="""
    SELECT *, log1p(usefulCount) AS log_utile, length(review) AS longueur,
           year(jour) AS annee, ... AS jour_semaine, ... AS nb_mots
    FROM __THIS__
  """
)
avec_variables = variables.transform(entrainement)
avec_variables.select(...).summary("min", "50%", "max").show()
avec_variables.groupBy(...).agg(F.avg("positif").alias("part_positifs")).orderBy(
  "jour_semaine"
).show()
quartiles = QuantileDiscretizer(
  numBuckets=..., inputCol="usefulCount", outputCol="utile_quartile"
).fit(...)
print(quartiles.getSplits())
quartiles.transform(entrainement).groupBy(...).agg(
  F.avg("positif").alias("part_positifs")
).orderBy("utile_quartile").show()
```

### Solution

In [ ]:
variables = SQLTransformer(
  statement="""
    SELECT *, log1p(usefulCount) AS log_utile, length(review) AS longueur,
           year(jour) AS annee, dayofweek(jour) AS jour_semaine,
           size(split(review, ' ')) AS nb_mots
    FROM __THIS__
  """
)
avec_variables = variables.transform(entrainement)
avec_variables.select("jour_semaine", "nb_mots").summary("min", "50%", "max").show()
avec_variables.groupBy("jour_semaine").agg(
  F.avg("positif").alias("part_positifs")
).orderBy("jour_semaine").show()
quartiles = QuantileDiscretizer(
  numBuckets=4, inputCol="usefulCount", outputCol="utile_quartile"
).fit(entrainement)
print(quartiles.getSplits())
quartiles.transform(entrainement).groupBy("utile_quartile").agg(
  F.avg("positif").alias("part_positifs")
).orderBy("utile_quartile").show()
# Les avis vont d'un mot à 1 857 (85 au milieu). La part d'avis positifs ne bouge
# presque pas d'un jour à l'autre (de 65,8 % à 67,3 %) : jour_semaine n'apporterait rien
# au modèle. Les quartiles coupent usefulCount à 6, 16 et 35, et la part d'avis
# positifs monte de 50 % dans la première classe à 84 % dans la dernière : les avis
# les plus lus sont plus souvent positifs, un effet que la classe rend utilisable
# par un modèle linéaire, même s'il n'était pas linéaire.

## 9. Variables catégorielles

Le code ci-dessous indexe et encode la condition, comme les diapositives : `indexeur` et `encodeur` sont les estimateurs, `modele_indexeur` et `modele_encodeur` les modèles qu'ils ont appris sur `entrainement`. Faites le même travail sur le médicament, `drugName`, vers `medicament_idx` puis `medicament_vec`.

Combien de dimensions ajouterait-il au vecteur des variables ? Combien de médicaments ont moins de cinq avis dans l'entraînement, et combien d'avis du test ont un médicament que l'entraînement n'a jamais vu ? Faut-il, d'après vous, ajouter `drugName` aux variables du modèle ?

In [ ]:
indexeur = StringIndexer(
  inputCol="condition", outputCol="condition_idx", handleInvalid="keep"
)
modele_indexeur = indexeur.fit(entrainement)
encodeur = OneHotEncoder(inputCol="condition_idx", outputCol="condition_vec")
modele_encodeur = encodeur.fit(modele_indexeur.transform(entrainement))
print(len(modele_indexeur.labelsArray[0]))

# Votre code ici

### Conseils supplémentaires

Comme pour la condition, `handleInvalid="keep"` donne aux médicaments inconnus l'indice qui suit le dernier appris, `len(medicaments)`, où `medicaments` est la liste `labelsArray[0]` du modèle. Un vecteur a l'attribut `size`. Les médicaments rares se comptent sur `entrainement.groupBy("drugName").count()`, filtré sur sa colonne `count`.

```python
indexeur_medicament = StringIndexer(
  inputCol="drugName", outputCol="medicament_idx", handleInvalid=...
)
modele_indexeur_medicament = indexeur_medicament.fit(...)
medicaments = modele_indexeur_medicament.labelsArray[0]
encodeur_medicament = OneHotEncoder(inputCol=..., outputCol="medicament_vec")
modele_encodeur_medicament = encodeur_medicament.fit(
  modele_indexeur_medicament.transform(entrainement)
)
encode = modele_encodeur_medicament.transform(
  modele_indexeur_medicament.transform(test)
)
print(len(medicaments), encode.first()["medicament_vec"].size)
print(entrainement.groupBy("drugName").count().filter(...).count())
print(encode.filter(F.col("medicament_idx") == ...).count())
```

### Solution

In [ ]:
indexeur_medicament = StringIndexer(
  inputCol="drugName", outputCol="medicament_idx", handleInvalid="keep"
)
modele_indexeur_medicament = indexeur_medicament.fit(entrainement)
medicaments = modele_indexeur_medicament.labelsArray[0]
encodeur_medicament = OneHotEncoder(
  inputCol="medicament_idx", outputCol="medicament_vec"
)
modele_encodeur_medicament = encodeur_medicament.fit(
  modele_indexeur_medicament.transform(entrainement)
)
encode = modele_encodeur_medicament.transform(
  modele_indexeur_medicament.transform(test)
)
print(len(medicaments), encode.first()["medicament_vec"].size)
print(entrainement.groupBy("drugName").count().filter("count < 5").count())
print(encode.filter(F.col("medicament_idx") == len(medicaments)).count())
# 2 967 médicaments, donc 2 967 dimensions de plus, quatre fois celles de la
# condition (750). La moitié (1 505) a moins de cinq avis dans l'entraînement : leurs
# colonnes n'apprendraient presque rien. 177 avis du test ont un médicament inconnu,
# qui reçoit un vecteur nul. Le médicament en dit sans doute plus que la condition sur
# la satisfaction : à essayer au chapitre suivant, en comparant le modèle avec et sans,
# les médicaments rares regroupés, ou par un TargetEncoder (Spark 4.0), une seule
# colonne au lieu de 2 967.

## 10. Pondérer et sélectionner les mots

Le code ci-dessous découpe les avis de `entrainement` en mots, sans les mots vides (`entrainement_mots`, gardé en cache), et en tire le TF-IDF, comme les diapositives : `frequences` et `tfidf` sont les estimateurs, `modele_frequences` et `modele_tfidf` leurs modèles, et `entrainement_tfidf` le résultat. Il affiche les dix mots les plus fréquents du vocabulaire.

Le modèle de l'`IDF` a un attribut `idf`, le poids de chaque mot, dans l'ordre de `modele_frequences.vocabulary` : quels sont les cinq mots de poids le plus faible, et les cinq de poids le plus fort ? Gardez ensuite, avec un `UnivariateFeatureSelector` appris sur `entrainement_tfidf`, les 1 000 mots les plus liés à `positif`, vers `tfidf_sel`. Affichez-en quelques-uns. Lesquels des vingt mots les plus fréquents la sélection écarte-t-elle, et pourquoi ?


**Documentation utile**

- [numpy.ndarray.argsort](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.argsort.html)
- [pyspark.ml.feature.UnivariateFeatureSelectorModel.selectedFeatures](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.UnivariateFeatureSelectorModel.html#pyspark.ml.feature.UnivariateFeatureSelectorModel.selectedFeatures)

In [ ]:
mots = RegexTokenizer(
  inputCol="review", outputCol="mots", pattern=r"[^\w']+", minTokenLength=2
)
utiles = StopWordsRemover(inputCol="mots", outputCol="mots_utiles")
entrainement_mots = utiles.transform(mots.transform(entrainement)).cache()
frequences = CountVectorizer(
  inputCol="mots_utiles", outputCol="tf", vocabSize=10000, minDF=5
)
modele_frequences = frequences.fit(entrainement_mots)
tfidf = IDF(inputCol="tf", outputCol="tfidf")
modele_tfidf = tfidf.fit(modele_frequences.transform(entrainement_mots))
entrainement_tfidf = modele_tfidf.transform(
  modele_frequences.transform(entrainement_mots)
)
print(modele_frequences.vocabulary[:10])

# Votre code ici

### Conseils supplémentaires

`modele_tfidf.idf.toArray()` fait des poids un tableau NumPy, dont `argsort()` rend les indices des mots du poids le plus faible au plus fort. Le sélecteur se règle comme dans les diapositives : des variables continues (`setFeatureType("continuous")`), une étiquette catégorielle (`setLabelType("categorical")`), donc un test ANOVA, et `setSelectionThreshold(1000)` en mode `numTopFeatures`. Son modèle a un attribut `selectedFeatures`, la liste des indices retenus.

```python
vocabulaire = modele_frequences.vocabulary
poids = modele_tfidf.idf.toArray()
ordre = poids.argsort()
print([vocabulaire[i] for i in ordre[:5]], poids[ordre[0]])
print([vocabulaire[i] for i in ordre[-5:]], poids[ordre[-1]])
selection = (
  UnivariateFeatureSelector(
    featuresCol=...,
    labelCol=...,
    outputCol="tfidf_sel",
    selectionMode="numTopFeatures",
  )
  .setFeatureType(...)
  .setLabelType(...)
  .setSelectionThreshold(...)
)
modele_selection = selection.fit(entrainement_tfidf)
retenus = modele_selection.selectedFeatures
print(len(retenus), [vocabulaire[i] for i in retenus[:15]])
print([mot for i, mot in enumerate(vocabulaire[:20]) if i not in ...])
```

### Solution

In [ ]:
vocabulaire = modele_frequences.vocabulary
poids = modele_tfidf.idf.toArray()
ordre = poids.argsort()
print([vocabulaire[i] for i in ordre[:5]], poids[ordre[0]])
print([vocabulaire[i] for i in ordre[-5:]], poids[ordre[-1]])
selection = (
  UnivariateFeatureSelector(
    featuresCol="tfidf",
    labelCol="positif",
    outputCol="tfidf_sel",
    selectionMode="numTopFeatures",
  )
  .setFeatureType("continuous")
  .setLabelType("categorical")
  .setSelectionThreshold(1000)
)
modele_selection = selection.fit(entrainement_tfidf)
retenus = modele_selection.selectedFeatures
print(len(retenus), [vocabulaire[i] for i in retenus[:15]])
print([mot for i, mot in enumerate(vocabulaire[:20]) if i not in retenus])
# Le poids le plus faible (1,29) va à side, puis à day, taking, effects et years,
# présents dans beaucoup d'avis ; le plus fort (9,62) aux mots les plus rares du
# vocabulaire, à la limite des 10 000 (kava, lastacaft, murmur…). La sélection garde
# 1 000 mots sur 10 000, dont des mots fréquents qui séparent les avis positifs des
# négatifs (side, effects, bad, anxiety…). Des vingt plus fréquents, elle écarte
# pain, get, time, back, one, weeks et medication : aussi présents dans les avis
# positifs que dans les négatifs, ils ne disent rien de la note.

## 11. Word2Vec et réduction de dimension

Entraînez un `Word2Vec` de `mots_utiles` vers `w2v`, en vecteurs de 50 dimensions, sur les mots présents au moins cinq fois, avec la graine 42. Son entraînement est lent : faites-le sur un échantillon de 40 % de `entrainement_mots` (graine 42), soit environ 36 000 avis, une à deux minutes sur Colab. Affichez les cinq mots les plus proches de `headache`, puis de `insomnia`.

Réduisez ensuite les vecteurs des avis de l'échantillon à 10 dimensions par une ACP (`PCA`, vers `w2v_acp`) : quelle part de la variance gardent-elles ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

`Word2Vec(inputCol=..., outputCol="w2v", vectorSize=..., minCount=..., seed=42)` ; son modèle a une méthode `findSynonyms(mot, nombre)`, qui rend un `DataFrame`. L'ACP s'entraîne sur la sortie du modèle (`transform`), et son modèle a un attribut `explainedVariance`, la part de la variance de chaque composante.

```python
echantillon = entrainement_mots.sample(fraction=..., seed=42)
plongements = Word2Vec(...).fit(echantillon)
plongements.findSynonyms("headache", 5).show()
plongements.findSynonyms("insomnia", 5).show()
acp = PCA(k=..., inputCol="w2v", outputCol="w2v_acp").fit(
  plongements.transform(echantillon)
)
print(acp.explainedVariance, sum(acp.explainedVariance))
```

### Solution

In [ ]:
echantillon = entrainement_mots.sample(fraction=0.4, seed=42)
plongements = Word2Vec(
  inputCol="mots_utiles", outputCol="w2v", vectorSize=50, minCount=5, seed=42
).fit(echantillon)
plongements.findSynonyms("headache", 5).show()
plongements.findSynonyms("insomnia", 5).show()
acp = PCA(k=10, inputCol="w2v", outputCol="w2v_acp").fit(
  plongements.transform(echantillon)
)
print(acp.explainedVariance, sum(acp.explainedVariance))
# Les voisins de headache sont d'autres effets secondaires (nausea, dizziness…),
# ceux de insomnia parlent de la nuit (nightmares, restlessness, sleeplessness) :
# des mots qui apparaissent dans les mêmes contextes. Les 10 composantes gardent
# environ deux tiers de la variance des 50 dimensions.

## 12. Le pipeline de préparation

Le nettoyage de l'exercice 7 est fait hors du pipeline : un job de scoring, qui appliquera le modèle à de nouveaux avis, devrait le recoder. Écrivez un `SQLTransformer`, `nettoyage`, qui en refait deux étapes sur des avis tels que lus : la condition invalide (les règles de l'exercice 4) ou manquante remplacée par `"inconnue"`, et la colonne `jour`. Il rend les colonnes du fichier, `condition` nettoyée, plus `jour`. Le décodage des avis, une UDF Python, reste hors du pipeline, comme le dédoublonnage, qui ne concerne que l'entraînement.

Réunissez ensuite les étages dans un `Pipeline`, `preparation` : `nettoyage` en tête, puis ceux des exercices 8 à 10 (`variables`, `indexeur`, `encodeur`, `mots`, `utiles`, `frequences` et `tfidf`), un `VectorAssembler` des trois variables numériques des diapositives vers `numeriques`, un `StandardScaler` centré vers `numeriques_std`, et un dernier `VectorAssembler` de `numeriques_std`, `condition_vec` et `tfidf` vers `features`. Les estimateurs y entrent sans `fit` : c'est le pipeline qui les entraîne.

Entraînez le pipeline sur `entrainement_brut`, les avis de l'entraînement tels que lus, textes décodés (le code ci-dessous le construit, comme `test_brut`). Appliquez le modèle obtenu à `test_brut`, et affichez la taille du vecteur `features` du premier avis et son nombre de valeurs non nulles. D'où vient cette taille ? Sauvegardez le modèle dans le dossier `modele-preparation`, rechargez-le, et vérifiez que le modèle rechargé donne le même vecteur au même avis. Affichez enfin, pour trois avis du test dont la condition lue est invalide ou manquante, leur `date`, puis `jour` et `condition` en sortie du modèle rechargé.


**Documentation utile**

- [CASE](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-case.html)
- [pyspark.ml.PipelineModel.load](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.PipelineModel.html#pyspark.ml.PipelineModel.load)

In [ ]:
entrainement_brut = avis.join(entrainement.select("id"), "id").withColumn(
  "review", decoder("review")
)
test_brut = avis.join(test.select("id"), "id").withColumn("review", decoder("review"))

# Votre code ici

### Conseils supplémentaires

La requête liste les colonnes du fichier une à une, sans `*`, pour que `condition` n'y soit qu'une fois : `CASE WHEN ... THEN 'inconnue' ELSE condition END AS condition`. En SQL, `condition LIKE '%</span>%'` teste une sous-chaîne, `condition RLIKE '^[a-z]'` une expression régulière, `condition IS NULL` une valeur manquante, et `to_date(date, 'MMMM d, yyyy')` lit la date. Les étages se donnent dans l'ordre où ils s'appliquent : chacun lit des colonnes qu'un étage précédent a écrites. Un vecteur a l'attribut `size` et la méthode `numNonzeros()`. `write().overwrite().save(dossier)` sauvegarde le modèle, `PipelineModel.load(dossier)` le recharge.

```python
nettoyage = SQLTransformer(
  statement="""
    SELECT id, drugName, review, rating, date, usefulCount,
           CASE WHEN ... THEN 'inconnue' ELSE condition END AS condition,
           ... AS jour
    FROM __THIS__
  """
)
preparation = Pipeline(
  stages=[
    nettoyage,
    variables,
    ...,
    VectorAssembler(inputCols=[...], outputCol="numeriques"),
    StandardScaler(inputCol="numeriques", outputCol="numeriques_std", withMean=...),
    VectorAssembler(inputCols=[...], outputCol="features"),
  ]
)
modele_preparation = preparation.fit(...)
premier = modele_preparation.transform(test_brut).select("features").first()["features"]
print(premier.size, premier.numNonzeros())
modele_preparation.write().overwrite().save("modele-preparation")
recharge = PipelineModel.load(...)
print(recharge.transform(test_brut).select("features").first()["features"] == premier)
recharge.transform(test_brut.filter(invalide | ...)).select(
  "date", "jour", "condition"
).show(3)
```

### Solution

In [ ]:
nettoyage = SQLTransformer(
  statement="""
    SELECT id, drugName, review, rating, date, usefulCount,
           CASE WHEN condition LIKE '%</span>%' OR condition RLIKE '^[a-z]'
                     OR condition IS NULL
                THEN 'inconnue' ELSE condition END AS condition,
           to_date(date, 'MMMM d, yyyy') AS jour
    FROM __THIS__
  """
)
preparation = Pipeline(
  stages=[
    nettoyage,
    variables,
    indexeur,
    encodeur,
    mots,
    utiles,
    frequences,
    tfidf,
    VectorAssembler(
      inputCols=["log_utile", "longueur", "annee"], outputCol="numeriques"
    ),
    StandardScaler(inputCol="numeriques", outputCol="numeriques_std", withMean=True),
    VectorAssembler(
      inputCols=["numeriques_std", "condition_vec", "tfidf"], outputCol="features"
    ),
  ]
)
modele_preparation = preparation.fit(entrainement_brut)
premier = modele_preparation.transform(test_brut).select("features").first()["features"]
print(premier.size, premier.numNonzeros())
modele_preparation.write().overwrite().save("modele-preparation")
recharge = PipelineModel.load("modele-preparation")
print(recharge.transform(test_brut).select("features").first()["features"] == premier)
recharge.transform(test_brut.filter(invalide | F.col("condition").isNull())).select(
  "date", "jour", "condition"
).show(3)
# 10 753 dimensions : 3 variables numériques, 750 pour la condition (751 catégories
# avec les inconnues, moins la dernière) et 10 000 mots. Un avis n'en remplit que
# quelques dizaines au plus : le vecteur est creux. Le modèle rechargé refait la même
# préparation, nettoyage de la condition et de la date compris : appliqué aux avis
# tels que lus, il ne demande au job de scoring que le décodage du texte.

## Fin

Arrêtez la session, qui libère les exécuteurs.

In [ ]:
spark.stop()